<a href="https://colab.research.google.com/github/rishabhucb/court-keypoints/blob/main/day5_new_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q roboflow
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, glob, yaml
from roboflow import Roboflow

PROJECT_DIR = '/content/drive/MyDrive/court_keypoints'
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("rishabh-ucb-gmail-com").project("hs-basketball-court-keypoints-hfzd3")

ds   = project.version(1).download("yolov8", location=f"{PROJECT_DIR}/hs_court_yolo")
coco = project.version(1).download("coco",   location=f"{PROJECT_DIR}/hs_court_coco")

print("\n--- data.yaml ---")
print(yaml.safe_load(open(f"{ds.location}/data.yaml")))

print("\n--- Keypoint IDs ---")
ann = json.load(open(glob.glob(f"{coco.location}/train/_annotations.coco.json")[0]))
for c in ann['categories']:
    if c.get('keypoints'):
        for i, name in enumerate(c['keypoints']):
            print(f"ID {i:2d} -> {name}")

In [ ]:
import cv2, glob, matplotlib.pyplot as plt

DATA2 = f'{PROJECT_DIR}/hs_court_yolo'
N_KPT = 18

def n_visible(lbl):
    v = list(map(float, open(lbl).readline().split()))[5:]
    return sum(1 for i in range(N_KPT) if v[3*i+2] > 0)

labels = sorted(glob.glob(f'{DATA2}/train/labels/*.txt'), key=n_visible, reverse=True)

for lbl in labels[:3]:                                   # the 3 images with the most visible points
    img_path = glob.glob(lbl.replace('/labels/', '/images/').rsplit('.', 1)[0] + '.*')[0]
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    v = list(map(float, open(lbl).readline().split()))[5:]
    for i in range(N_KPT):
        if v[3*i+2] > 0:
            x, y = int(v[3*i]*w), int(v[3*i+1]*h)
            cv2.circle(img, (x, y), 6, (255, 0, 0), -1)
            cv2.putText(img, str(i), (x+8, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 3)
    print(lbl.split('/')[-1][:40], "-", n_visible(lbl), "visible points")
    plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis('off'); plt.show()

In [ ]:
for want in [3, 4, 5]:
    for lbl in labels:
        v = list(map(float, open(lbl).readline().split()))[5:]
        if v[3*want+2] > 0:
            img_path = glob.glob(lbl.replace('/labels/', '/images/').rsplit('.', 1)[0] + '.*')[0]
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            for i in range(N_KPT):
                if v[3*i+2] > 0:
                    x, y = int(v[3*i]*w), int(v[3*i+1]*h)
                    cv2.circle(img, (x, y), 6, (255, 0, 0), -1)
                    cv2.putText(img, str(i), (x+8, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 3)
            plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis('off')
            plt.title(f'Image where ID {want} is visible'); plt.show()
            break
    else:
        print(f"ID {want} is never visible in train")

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import random

old_model = YOLO(f'{PROJECT_DIR}/runs/baseline_v4/weights/last.pt')
test_imgs = random.sample(glob.glob(f'{DATA2}/valid/images/*'), 3)

for r in old_model.predict(test_imgs, conf=0.25, verbose=False):
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.show()

In [ ]:
!pip install -q ultralytics imagehash
from google.colab import drive
drive.mount('/content/drive')

import glob, os, shutil, yaml, imagehash
from PIL import Image

PROJECT_DIR = '/content/drive/MyDrive/court_keypoints'
SRC = f'{PROJECT_DIR}/hs_court_yolo'
DST = f'{PROJECT_DIR}/hs_court_clean'

# 1. Collect all images + labels
items = []
for split in ['train', 'valid', 'test']:
    for img in glob.glob(f'{SRC}/{split}/images/*'):
        lbl = f'{SRC}/{split}/labels/' + os.path.splitext(os.path.basename(img))[0] + '.txt'
        if os.path.exists(lbl):
            items.append((img, lbl))
print(len(items), "images")

# 2. Group near-duplicate frames
fps = [imagehash.phash(Image.open(i)) for i, _ in items]
parent = list(range(len(items)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        if fps[i] - fps[j] <= 6:
            parent[find(i)] = find(j)
groups = {}
for i in range(len(items)):
    groups.setdefault(find(i), []).append(i)
groups = sorted(groups.values(), key=len, reverse=True)
print("Groups:", len(groups), "| largest sizes:", [len(g) for g in groups[:10]])

# 3. Give whole groups to train / valid / test (~75 / 15 / 10)
target = {'train': 0.75, 'valid': 0.15, 'test': 0.10}
count = {k: 0 for k in target}
assign = {k: [] for k in target}
for g in groups:
    s = max(target, key=lambda k: target[k] * len(items) - count[k])
    assign[s] += g; count[s] += len(g)

# 4. Copy with cleaned labels (off-screen points -> not visible, box clipped to image)
def clean(line):
    v = line.split()
    cx, cy, w, h = map(float, v[1:5]); kp = [float(x) for x in v[5:]]
    x1, y1, x2, y2 = max(cx - w/2, 0), max(cy - h/2, 0), min(cx + w/2, 1), min(cy + h/2, 1)
    for i in range(0, len(kp), 3):
        if not (0 <= kp[i] <= 1 and 0 <= kp[i+1] <= 1):
            kp[i:i+3] = [0.0, 0.0, 0.0]
    return ' '.join([v[0]] + [f'{x:.6f}' for x in [(x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1] + kp])

shutil.rmtree(DST, ignore_errors=True)
for split, idx in assign.items():
    os.makedirs(f'{DST}/{split}/images'); os.makedirs(f'{DST}/{split}/labels')
    for i in idx:
        img, lbl = items[i]
        shutil.copy(img, f'{DST}/{split}/images/')
        lines = [clean(l) for l in open(lbl) if l.strip()]
        open(f'{DST}/{split}/labels/' + os.path.basename(lbl), 'w').write('\n'.join(lines) + '\n')
    print(split, len(idx), "images")

# 5. data.yaml with the correct flip_idx (mirror swaps the two baskets)
yaml.safe_dump({'path': DST, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
                'nc': 1, 'names': ['basketball'], 'kpt_shape': [18, 3],
                'flip_idx': [10, 11, 12, 13, 14, 15, 16, 17, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7]},
               open(f'{DST}/data.yaml', 'w'))
print(open(f'{DST}/data.yaml').read())

In [ ]:
from ultralytics import YOLO

DATA_HS = f'{PROJECT_DIR}/hs_court_clean/data.yaml'

model = YOLO('yolo11n-pose.pt')
model.train(
    data=DATA_HS,
    epochs=150,
    imgsz=640,
    batch=16,
    fliplr=0.5,          # safe now that flip_idx is correct
    patience=50,         # don't stop too early (lesson from v1)
    project=f'{PROJECT_DIR}/runs',
    name='hs_v1',
)

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
from ultralytics import YOLO

PROJECT_DIR = '/content/drive/MyDrive/court_keypoints'
DATA_HS = f'{PROJECT_DIR}/hs_court_clean/data.yaml'

hs = YOLO(f'{PROJECT_DIR}/runs/hs_v1/weights/last.pt')
te = hs.val(data=DATA_HS, split='test')
print("TEST pose mAP50:   ", round(te.pose.map50, 3))
print("TEST pose mAP50-95:", round(te.pose.map, 3))

In [ ]:
import glob, random, cv2
import matplotlib.pyplot as plt

new_gym = random.sample(glob.glob(f'{PROJECT_DIR}/hs_court_clean/test/images/*'), 3)
old_gym = random.sample(glob.glob(f'{PROJECT_DIR}/court_v4/test/images/*'), 2)

for title, imgs in [("TEST images (same gyms as training)", new_gym),
                    ("NEVER-SEEN gym (old dataset)", old_gym)]:
    for r in hs.predict(imgs, conf=0.25, verbose=False):
        plt.figure(figsize=(16, 9))
        plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        plt.title(title, fontsize=14); plt.axis('off'); plt.show()